In [136]:
import pandas as pd
import ir_datasets

human_qrels = (
    pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").qrels_iter())
    .astype({"query_id": str, "doc_id": str})
    .drop("iteration", axis="columns")
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
)

adapter_qrels_256 = (
    pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
    .astype({"query_id": str, "doc_id": str})
    .assign(
        dl20=lambda df: df["query_id"].isin(human_qrels["query_id"].unique()),
        relevance=lambda df: (df["relevance"] > 0.5).astype(int),
    )
    .query("dl20 == True")
    .drop("dl20", axis="columns")
)

adapter_qrels_192 = (
    pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
    .astype({"query_id": str, "doc_id": str})
    .assign(
        dl20=lambda df: df["query_id"].isin(human_qrels["query_id"].unique()),
        relevance=lambda df: (df["relevance"] > 0.5).astype(int),
    )
    .query("dl20 == True")
    .drop("dl20", axis="columns")
)

In [137]:
# Identify which qrels were used for training (the ones not in the predicted qrels)
# so we can add them to other qrel source to level the playing field

train_qrels = (
    pd.concat([
        (
            pd.concat([human_qrels, adapter_qrels_256])
            .drop_duplicates(keep=False)
        ),
        (
            pd.concat([human_qrels, adapter_qrels_192])
            .drop_duplicates(keep=False)
        )
    ]).drop_duplicates()
)


In [127]:
zero_qrels = pd.concat([
    (
        adapter_qrels_256
        .assign(relevance=0)
    ),
    train_qrels
])

adapter_qrels_256 = pd.concat([adapter_qrels_256, train_qrels])
adapter_qrels_192 = pd.concat([adapter_qrels_192, train_qrels])

gpt4_umbrela_qrels = (
    pd.concat([
        (
            pd.read_csv("../data/external/dl20-llm-qrels/gpt-4o_binary_dl20-umbrela.txt", header=None, names=["query_id", "q0", "doc_id", "relevance"], sep=" ")
            .astype({"query_id": str, "doc_id": str})
            .drop("q0", axis="columns")
        ),
        train_qrels
    ])
    .drop_duplicates(keep="last")
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
)

gpt4_human_qrels = (
    pd.concat([
        (
            pd.read_csv("../data/external/dl20-llm-qrels/gpt-4o_binary_dl20-human.txt", header=None, names=["query_id", "q0", "doc_id", "relevance"], sep=" ")
            .astype({"query_id": str, "doc_id": str})
            .drop("q0", axis="columns")
        ),
        train_qrels
    ])
    .drop_duplicates(keep="last")
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
)

In [152]:
few_shot_oss_8 = (
    pd.concat([
        (
            pd.read_json("gpt-oss-qrels.json", lines=True)
            .query("prompt == 'few_shot_8'")
            .drop("prompt", axis=1)
        ),
        train_qrels
    ])
    .drop_duplicates(keep="last")
    .assign(relevance=lambda df: (df["relevance"].astype(int) > 0.5).astype(int))
)

few_shot_oss_16 = (
    pd.concat([
        (
            pd.read_json("gpt-oss-qrels.json", lines=True)
            .query("prompt == 'few_shot_16'")
            .drop("prompt", axis=1)
        ),
        train_qrels
    ])
    .drop_duplicates(keep="last")
    .assign(relevance=lambda df: (df["relevance"].astype(int) > 0.5).astype(int))
)

few_shot_oss_32 = (
    pd.concat([
        (
            pd.read_json("gpt-oss-qrels.json", lines=True)
            .query("prompt == 'few_shot_32'")
            .drop("prompt", axis=1)
        ),
        train_qrels
    ])
    .drop_duplicates(keep="last")
    .assign(relevance=lambda df: (df["relevance"].astype(int) > 0.5).astype(int))
)

topics = pd.read_json("gpt-oss-qrels.json", lines=True)["query_id"].unique().astype(str)
topics

array(['1131069', '330975', '324585', '701453', '1136047', '768208',
       '1043135', '332593', '555530', '1132532', '336901', '1108651'],
      dtype='<U21')

In [151]:
data

array([], shape=(2, 0), dtype=int64)

In [153]:
import krippendorff

df_reliability = []

for (qrel_data, label) in [
    (zero_qrels, "baseline_0"),
    (adapter_qrels_192, "adapter-192"),
    (adapter_qrels_256, "adapter-256"),
    (gpt4_human_qrels, "gpt4-human-prompt"),
    (gpt4_umbrela_qrels, "gpt4-umbrela-prompt"),
    (few_shot_oss_8, "few-shot-8"),
    (few_shot_oss_16, "few-shot-16"),
    (few_shot_oss_32, "few-shot-32"),
]:
    data = (
        pd.concat([
            human_qrels,
            train_qrels
        ]).drop_duplicates(keep=False)
        .query("query_id in @topics")
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .merge(qrel_data.assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int)), on=["query_id", "doc_id"])
        .loc[:, ["relevance_x", "relevance_y"]]
        .values
        .T
    )
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="nominal", value_domain=(0, 1))
    df_reliability.append({
        "qrels": label,
        "alpha": alpha,
    })

df_reliability = pd.DataFrame(df_reliability)
df_reliability

,qrels,alpha
0,baseline_0,-0.298729
1,adapter-192,0.722951
2,adapter-256,0.722951
3,gpt4-human-prompt,0.126553
4,gpt4-umbrela-prompt,0.157996
5,few-shot-8,-0.284696
6,few-shot-16,-0.284696
7,few-shot-32,-0.284696


In [131]:
df_reliability.groupby("qrels").mean()

,alpha
qrels,
adapter-192,0.574195
adapter-256,0.574195
baseline_0,-0.303191
gpt4-human-prompt,0.038674
gpt4-umbrela-prompt,0.076187


In [156]:
import trectools
from glob import glob
from tqdm.notebook import tqdm

runs = [trectools.TrecRun(f) for f in glob(f"../data/external/dl20-passages-runs/*.gz")]
depths = [5, 10, 20, 50]
eval_data = []

for (qrel_data, label) in [
    (zero_qrels, "baseline_0"),
    (adapter_qrels_192, "adapter-192"),
    (adapter_qrels_256, "adapter-256"),
    (gpt4_human_qrels, "gpt4-human-prompt"),
    (gpt4_umbrela_qrels, "gpt4-umbrela-prompt"),
    (human_qrels, "ground-truth"),
    (few_shot_oss_8, "few-shot-8"),
    (few_shot_oss_16, "few-shot-16"),
    (few_shot_oss_32, "few-shot-32"),
]:
    qrels = trectools.TrecQrel()
    qrels.qrels_data = qrel_data.rename({"query_id": "query", "doc_id": "docid", "relevance": "rel"}, axis="columns").query("query in @topics")
    for r in tqdm(runs, desc=label):
        eval = trectools.TrecEval(r, qrels)
        eval_data.append(
            pd.concat([
                pd.concat([eval.get_ndcg(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat(
                    [eval.get_unjudged(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"})
            ])
            .reset_index()
            .assign(
                system=r.get_runid(),
                source=label
            )
        )

df = pd.concat(eval_data)

baseline_0:   0%|          | 0/59 [00:00<?, ?it/s]

adapter-192:   0%|          | 0/59 [00:00<?, ?it/s]

adapter-256:   0%|          | 0/59 [00:00<?, ?it/s]

gpt4-human-prompt:   0%|          | 0/59 [00:00<?, ?it/s]

gpt4-umbrela-prompt:   0%|          | 0/59 [00:00<?, ?it/s]

ground-truth:   0%|          | 0/59 [00:00<?, ?it/s]

few-shot-8:   0%|          | 0/59 [00:00<?, ?it/s]

few-shot-16:   0%|          | 0/59 [00:00<?, ?it/s]

few-shot-32:   0%|          | 0/59 [00:00<?, ?it/s]

In [157]:
(
    df
    .drop(["query", "index"], axis="columns")
    .dropna(subset="value")
    .astype({"value": float, "system": str, "source": str})
    .reset_index(drop=True)
    .groupby(["metric", "system", "source"])
    .mean()
    .reset_index()
    .pivot(index=["system", "metric"], columns="source", values="value")
    .reset_index()
    .set_index("system")
    .groupby("metric")
    .corr(method="spearman")
    .reset_index()
    .query("source == 'ground-truth'")
    .drop(["source", "ground-truth"], axis=1)
    .query("metric in ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']")
    .set_index("metric")
    .transpose()
    .loc[:, ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']]
)

metric,NDCG@5,NDCG@10,NDCG@20,NDCG@50
source,,,,
adapter-192,0.887798,0.969913,0.983182,0.972574
adapter-256,0.887798,0.969913,0.983182,0.972574
baseline_0,0.747311,0.717457,0.911453,0.903533
few-shot-16,0.747311,0.717457,0.911453,0.903533
few-shot-32,0.747311,0.717457,0.911453,0.903533
few-shot-8,0.747311,0.717457,0.911453,0.903533
gpt4-human-prompt,0.877481,0.904449,0.964655,0.988501
gpt4-umbrela-prompt,0.885898,0.894659,0.961878,0.980377


In [167]:
print(few_shot_oss_8.query("query_id in @topics")["relevance"].value_counts())

relevance
0    1250
1     127
Name: count, dtype: int64


In [166]:
print(zero_qrels.query("query_id in @topics")["relevance"].value_counts())

relevance
0    3016
1     127
Name: count, dtype: int64


In [168]:
few_shot_oss_8

,query_id,doc_id,relevance
0,1131069,8772624,0
9,1131069,2723544,0
18,1131069,1780970,0
27,1131069,2996190,0
36,1131069,5538323,0
...,...,...,...
376,121171,2297409,0
381,121171,4355386,0
390,121171,5809828,0
391,121171,993579,0
